---
title: Land area analysis
date: now
author: Jan Cap
---

In [ ]:
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load_transform()
municipalities

## Data alignment

In [ ]:
from geoscore_de.data_flow.features.area import AreaFeature

df = AreaFeature(
    raw_data_path="../../data/raw/features/33111-01-03-5-land-area.csv",
    tform_data_path="../../data/tform/features/area.csv",
).load()
df

Original data contain multiple columns for land area coverage. Area is in absolute values in hectares. We will keep all columns, and normalize them by total land area to get relative values.
Our main focus will be on settlement area, traffic area, vegetation area, water area which are separated into many subcategories.

In [ ]:
df.columns

In [ ]:
from geoscore_de.data_flow.features.area import AreaFeature

df = AreaFeature(
    raw_data_path="../../data/raw/features/33111-01-03-5-land-area.csv",
    tform_data_path="../../data/tform/features/area.csv",
).load_transform()
df

In [ ]:
df_merged = df.merge(municipalities, on="AGS", how="outer", indicator=True)
df_merged["_merge"].value_counts()

There is 3138 extra records in the land area dataset, but no missing values in the municipalities dataset. Extra records are likely due to records of different administrative levels, e.g. districts, which we will ignore.

In [ ]:
df_merged = df.merge(municipalities, on="AGS", how="right")
df_merged

In [ ]:
# count of missing values in each column
df_merged.isna().sum().sort_values()

## Visualizations

In [ ]:
# now show it on map of germany
from geoscore_de.data_flow.geo import load_geo_data

gdf = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
# join data by ags
gdf_merged = gdf.merge(df, on="AGS", how="outer", indicator=True)

In [ ]:
import matplotlib.pyplot as plt

cols = {
    "total_settlement_area": "Total Settlement Area",
    "total_traffic_area": "Total Traffic Area",
    "total_vegetation_area": "Total Vegetation Area",
    "total_water_area": "Total Water Area",
}

for col, title in cols.items():
    gdf_merged.plot(
        column=col,
        legend=True,
        cmap="YlOrRd",
        figsize=(12, 8),
        # vmin=0,
        # vmax=1,
        missing_kwds={"color": "gray"},
    )
    plt.title(title + " in German Municipalities")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="total_water_area",
    legend=True,
    cmap="YlOrRd",
    figsize=(12, 8),
    # vmin=0,
    vmax=0.5,
    missing_kwds={"color": "gray"},
)
plt.title("Total Water Area in German Municipalities")
plt.show()

We can see that bigger cities have more settlement and traffic area, while rural areas have more vegetation area. Water area is specific, because it follows the river and lake distribution, which is not related to urbanization.
Traffic area is less extreme than settlement area.